<a href="https://colab.research.google.com/github/Boonyaratt/Water-Quality-Assessment/blob/main/notebooks/02_dataset_assembly/02_Merge_rswq_dataset.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from pathlib import Path
import hashlib
import json

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 160)
pd.set_option("display.max_row", 160)

In [ ]:
# Default paths for the files currently in E:/Downloads.
# If running in Colab, upload/copy the CSVs and change BASE accordingly.
BASE = Path(r"/content")

S2_FULL_CSV = BASE / "sentinel2_model_dataset_epo14_2562_2568_b200_watermask_v3_s2formula.csv"
S2_STRICT_CSV = BASE / "sentinel2_train_strict_candidates_epo14_2562_2568_b200_watermask_v3_s2formula.csv"
S2_RELAXED_CSV = BASE / "sentinel2_train_relaxed_candidates_epo14_2562_2568_b200_watermask_v3_s2formula.csv"
CONTEXT_CSV = BASE / "rswq_context_features_ctx_2562_2568_gsmap_era5h_strict_v2.csv"

OUT_DIR = BASE / "rswq_tabular_assembly_outputs"
OUT_DIR.mkdir(parents=True, exist_ok=True)

YEAR_LABEL = "2562_2568"

print("Output directory:", OUT_DIR)
for path in [S2_FULL_CSV, S2_STRICT_CSV, S2_RELAXED_CSV, CONTEXT_CSV]:
    print(path.name, "exists:", path.exists())


Output directory: /content/rswq_tabular_assembly_outputs
sentinel2_model_dataset_epo14_2562_2568_b200_watermask_v3_s2formula.csv exists: True
sentinel2_train_strict_candidates_epo14_2562_2568_b200_watermask_v3_s2formula.csv exists: True
sentinel2_train_relaxed_candidates_epo14_2562_2568_b200_watermask_v3_s2formula.csv exists: True
rswq_context_features_ctx_2562_2568_gsmap_era5h_strict_v2.csv exists: True


In [ ]:
from google.colab import files
uploaded = files.upload()

Saving rswq_context_features_ctx_2562_2568_gsmap_era5h_strict_v2.csv to rswq_context_features_ctx_2562_2568_gsmap_era5h_strict_v2.csv


In [ ]:
def load_csv(path):
  if not path.exists():
    raise FileNotFoundError(path)
  return pd.read_csv(path)

s2_full = load_csv(S2_FULL_CSV)
s2_strict = load_csv(S2_STRICT_CSV)
s2_relaxed = load_csv(S2_RELAXED_CSV)
context = load_csv(CONTEXT_CSV)

print("s2_full:", s2_full.shape)
print("s2_strict:", s2_strict.shape)
print("s2_relaxed:", s2_relaxed.shape)
print("context:", context.shape)

s2_full: (826, 109)
s2_strict: (149, 109)
s2_relaxed: (390, 109)
context: (826, 129)


In [ ]:
def check_sample_id(df, name):
    if "sample_id" not in df.columns:
        raise KeyError(f"{name} has no sample_id column")
    print(
        name,
        "rows:", len(df),
        "unique sample_id:", df["sample_id"].nunique(),
        "duplicates:", int(df["sample_id"].duplicated().sum()),
    )
    if df["sample_id"].duplicated().any():
        raise ValueError(f"{name} has duplicated sample_id")


check_sample_id(s2_full, "s2_full")
check_sample_id(s2_strict, "s2_strict")
check_sample_id(s2_relaxed, "s2_relaxed")
check_sample_id(context, "context")

full_ids = set(s2_full["sample_id"].astype(str))
context_ids = set(context["sample_id"].astype(str))

print("full/context sample_id intersection:", len(full_ids & context_ids))
print("full/context exact same set:", full_ids == context_ids)


s2_full rows: 826 unique sample_id: 826 duplicates: 0
s2_strict rows: 149 unique sample_id: 149 duplicates: 0
s2_relaxed rows: 390 unique sample_id: 390 duplicates: 0
context rows: 826 unique sample_id: 826 duplicates: 0
full/context sample_id intersection: 826
full/context exact same set: True


In [ ]:
def merge_s2_with_context(s2_df, context_df):
    # Drop duplicate metadata columns from context to avoid _x/_y suffixes.
    # Keep the Sentinel-2/Ground Observation table as the main metadata source.
    duplicate_context_cols = sorted((set(s2_df.columns) & set(context_df.columns)) - {"sample_id"})
    context_to_add = context_df.drop(columns=duplicate_context_cols)

    print("Duplicate context metadata columns dropped:", duplicate_context_cols)
    print("Context columns added:", len(context_to_add.columns) - 1)

    merged = s2_df.merge(
        context_to_add,
        on="sample_id",
        how="left",
        validate="one_to_one",
    )
    return merged


In [ ]:
model_input_full = merge_s2_with_context(s2_full, context)
model_input_strict = merge_s2_with_context(s2_strict, context)
model_input_relaxed = merge_s2_with_context(s2_relaxed, context)

print("model_input_full:", model_input_full.shape)
print("model_input_strict:", model_input_strict.shape)
print("model_input_relaxed:", model_input_relaxed.shape)


Duplicate context metadata columns dropped: ['canonical_station_survey_id', 'coordinate_assignment_status', 'coordinate_waterbody', 'field_date', 'station_canonical', 'station_original', 'station_survey_id', 'survey_no', 'water_body', 'year_ad', 'year_be']
Context columns added: 117
Duplicate context metadata columns dropped: ['canonical_station_survey_id', 'coordinate_assignment_status', 'coordinate_waterbody', 'field_date', 'station_canonical', 'station_original', 'station_survey_id', 'survey_no', 'water_body', 'year_ad', 'year_be']
Context columns added: 117
Duplicate context metadata columns dropped: ['canonical_station_survey_id', 'coordinate_assignment_status', 'coordinate_waterbody', 'field_date', 'station_canonical', 'station_original', 'station_survey_id', 'survey_no', 'water_body', 'year_ad', 'year_be']
Context columns added: 117
model_input_full: (826, 226)
model_input_strict: (149, 226)
model_input_relaxed: (390, 226)


In [ ]:
for name, df in {
    "full": model_input_full,
    "strict": model_input_strict,
    "relaxed": model_input_relaxed,
}.items():
    print("\n==", name, "==")
    print("shape:", df.shape)
    print("unique sample_id:", df["sample_id"].nunique())
    print("duplicate sample_id:", int(df["sample_id"].duplicated().sum()))
    if "match_window" in df.columns:
        print("match_window:", df["match_window"].value_counts(dropna=False).to_dict())
    if "data_quality" in df.columns:
        print("data_quality:", df["data_quality"].value_counts(dropna=False).to_dict())

display(model_input_full.head())



== full ==
shape: (826, 226)
unique sample_id: 826
duplicate sample_id: 0
match_window: {'within_3_days': 254, 'fallback_7d': 233, 'within_1_day': 185, 'exact_date': 99, 'no_match': 55}
data_quality: {'has_water_pixel': 431, 'matched_no_clear_pixel': 200, 'clear_but_no_water_pixel': 140, 'no_match': 55}

== strict ==
shape: (149, 226)
unique sample_id: 149
duplicate sample_id: 0
match_window: {'within_3_days': 67, 'within_1_day': 61, 'exact_date': 21}
data_quality: {'has_water_pixel': 149}

== relaxed ==
shape: (390, 226)
unique sample_id: 390
duplicate sample_id: 0
match_window: {'fallback_7d': 131, 'within_3_days': 123, 'within_1_day': 95, 'exact_date': 41}
data_quality: {'has_water_pixel': 390}


,sample_id,turbidity_NTU_clean,SS_mg_l_clean,DO_mg_l_clean,BOD_mg_l_clean,NH3_N_mg_l_clean,TCB_MPN_100ml_clean,FCB_MPN_100ml_clean,WQI_clean,WQI_DO_score_pcd5,WQI_BOD_score_pcd5,WQI_TCB_score_pcd5,WQI_FCB_score_pcd5,WQI_NH3_score_pcd5,WQI_reported,WQI_score_input_complete,WQI_mean_subindex_pcd5,WQI_mean_level_pcd5,WQI_worst_parameter_level_pcd5,WQI_adjustment_pcd5,WQI_recalc_pcd5,WQI_source,WQI_recalc_status,pH_clean,conductivity_clean,salinity_ppt_clean,temp_air_clean,temp_water_clean,station_survey_id,canonical_station_survey_id,station_original,station_canonical,survey_no,year_be,year_ad,field_date,lat,lon,water_body,coordinate_waterbody,coordinate_assignment_status,sat_date,days_diff,days_diff_signed,match_type,image_id,scene_cloud_pct,buffer_m,scale_m,primary_window_days,fallback_window_days,valid_pixel_count,valid_water_pixel_count,total_pixel_count,water_pixel_fraction,local_cloud_fraction,selection_reason,candidate_valid_count,candidate_water_count,candidate_has_enough_water,local_match_rank,min_candidate_water_pixels,B1_median,B2_median,B3_median,B4_median,B5_median,B6_median,B7_median,B8_median,B8A_median,B9_median,B11_median,B12_median,NDWI_median,MNDWI_median,NDTI_median,NDCI_median,NDVI_median,AWEIsh_median,...,rain_gsmap_max_prior_336h_mm_h,rainy_hours_gsmap_prior_336h,rain_gsmap_prior_720h_mm,rain_gsmap_max_prior_720h_mm_h,rainy_hours_gsmap_prior_720h,hours_since_last_rain_gsmap_mean,era5h_air_temp_prior_1h_C,era5h_dewpoint_prior_1h_C,era5h_wind_mean_prior_1h_m_s,era5h_wind_max_prior_1h_m_s,era5h_runoff_prior_1h_mm,era5h_solar_prior_1h_MJ_m2,era5h_air_temp_prior_3h_C,era5h_dewpoint_prior_3h_C,era5h_wind_mean_prior_3h_m_s,era5h_wind_max_prior_3h_m_s,era5h_runoff_prior_3h_mm,era5h_solar_prior_3h_MJ_m2,era5h_air_temp_prior_6h_C,era5h_dewpoint_prior_6h_C,era5h_wind_mean_prior_6h_m_s,era5h_wind_max_prior_6h_m_s,era5h_runoff_prior_6h_mm,era5h_solar_prior_6h_MJ_m2,era5h_air_temp_prior_12h_C,era5h_dewpoint_prior_12h_C,era5h_wind_mean_prior_12h_m_s,era5h_wind_max_prior_12h_m_s,era5h_runoff_prior_12h_mm,era5h_solar_prior_12h_MJ_m2,era5h_air_temp_prior_24h_C,era5h_dewpoint_prior_24h_C,era5h_wind_mean_prior_24h_m_s,era5h_wind_max_prior_24h_m_s,era5h_runoff_prior_24h_mm,era5h_solar_prior_24h_MJ_m2,era5h_air_temp_prior_72h_C,era5h_dewpoint_prior_72h_C,era5h_wind_mean_prior_72h_m_s,era5h_wind_max_prior_72h_m_s,era5h_runoff_prior_72h_mm,era5h_solar_prior_72h_MJ_m2,era5h_air_temp_prior_168h_C,era5h_dewpoint_prior_168h_C,era5h_wind_mean_prior_168h_m_s,era5h_wind_max_prior_168h_m_s,era5h_runoff_prior_168h_mm,era5h_solar_prior_168h_MJ_m2,jrc_occurrence_mean_500m,jrc_seasonality_mean_500m,jrc_recurrence_mean_500m,srtm_elevation_mean_500m,srtm_slope_mean_500m,wc_tree_frac_500m,wc_shrub_frac_500m,wc_grass_frac_500m,wc_crop_frac_500m,wc_built_frac_500m,wc_bare_frac_500m,wc_water_frac_500m,wc_wetland_frac_500m,wc_mangrove_frac_500m,wc_tree_frac_1000m,wc_shrub_frac_1000m,wc_grass_frac_1000m,wc_crop_frac_1000m,wc_built_frac_1000m,wc_bare_frac_1000m,wc_water_frac_1000m,wc_wetland_frac_1000m,wc_mangrove_frac_1000m,wc_tree_frac_5000m,wc_shrub_frac_5000m,wc_grass_frac_5000m,wc_crop_frac_5000m,wc_built_frac_5000m,wc_bare_frac_5000m,wc_water_frac_5000m,wc_wetland_frac_5000m,wc_mangrove_frac_5000m
0,LS01_3_62_20190722,8.0,16.0,7.0,0.2,0.16,9200.0,1400.0,69.0,83.0810,96.1334,67.893,69.713,78.90880,NaN,True,79.145840,0.0,1.0,10.0,69.0,recalculated_pcd5,recalculated_from_complete_pcd5_inputs,7.2,2020.0,1.0,32.8,28.7,LS01_3_62,LS01_3_62,LS01,LS01,3_62,2562,2019,2019-07-22,9.940972,99.148500,หลังสวน,แม่น้ำหลังสวนตอนล่าง,by_station_code,2019-07-21,0.836390,-0.836390,window_3d,20190721T033549_20190721T035429_T47PNM,61.884119,200,20,3,7,99,0,308,0.000000,0.678571,primary_no_enough_water,81.0,0.0,0.0,1.000898e+06,3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,10.766462,52.812693,291.529122,10.766462,78.232198,57.185759,30.096657,24.128915,1.026316,1.026316,0.000000,2.960443,29.108698,24.331302,1.054771,1.193210,0.000000,6.8

In [ ]:
full_out = OUT_DIR / f"rswq_model_input_full_s2_context_{YEAR_LABEL}.csv"
strict_out = OUT_DIR / f"rswq_model_input_strict_s2_context_{YEAR_LABEL}.csv"
relaxed_out = OUT_DIR / f"rswq_model_input_relaxed_s2_context_{YEAR_LABEL}.csv"

model_input_full.to_csv(full_out, index=False, encoding="utf-8-sig")
model_input_strict.to_csv(strict_out, index=False, encoding="utf-8-sig")
model_input_relaxed.to_csv(relaxed_out, index=False, encoding="utf-8-sig")

metadata = {
    "inputs": {
        "s2_full": str(S2_FULL_CSV),
        "s2_strict": str(S2_STRICT_CSV),
        "s2_relaxed": str(S2_RELAXED_CSV),
        "context": str(CONTEXT_CSV),
    },
    "outputs": {
        "full": str(full_out),
        "strict": str(strict_out),
        "relaxed": str(relaxed_out),
    },
    "merge_key": "sample_id",
    "note": "Simple assembly only. No split labels, flags, imputation, scaling, or extra feature engineering added.",
}

metadata_out = OUT_DIR / f"rswq_model_input_simple_assembly_metadata_{YEAR_LABEL}.json"
metadata_out.write_text(json.dumps(metadata, ensure_ascii=False, indent=2), encoding="utf-8")

for path in [full_out, strict_out, relaxed_out, metadata_out]:
    print(path, "size:", path.stat().st_size)


/content/rswq_tabular_assembly_outputs/rswq_model_input_full_s2_context_2562_2568.csv size: 2488428
/content/rswq_tabular_assembly_outputs/rswq_model_input_strict_s2_context_2562_2568.csv size: 482081
/content/rswq_tabular_assembly_outputs/rswq_model_input_relaxed_s2_context_2562_2568.csv size: 1259650
/content/rswq_tabular_assembly_outputs/rswq_model_input_simple_assembly_metadata_2562_2568.json size: 908
